In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Add new network MVRD first

In [4]:
# from pycds import Network
# # help(Network)

# net = Network(name="MVRD", publish=False)

# session.add(net)
# session.flush()   # 👈 INSERT happens here
# session.commit()
# # print(net.id)  # or net.id depending on model

## Deal with the "additional" stations, which belongs to the network

Two items are excluded, line 2035 (FERN) has been inserted, and 2085, don't know what PC-FWX is (BC-FWX?)

In [5]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_addition = df[
    (df["ISSUE"].str.strip().str.lower() == "additional") 
]
df_addition = df_addition[df_addition["Network ID"].notna()]
df_addition
df_pcds = df_addition[['Network ID' ,'Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_ted = df_addition[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)


df_pcds['Station_name'] = df_addition['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Unique Names'])

df_pcds['native_id'] = df_addition['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Native ID'])


In [6]:
import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df_pcds[['lat', 'lon', 'Elevation']] = df_pcds['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df_pcds = df_pcds.drop(columns=['Unique Location (String)'])

In [7]:
df_pcds.iloc[60:120]

# df_ted.head(10)

,Network ID,Station ID,Station_name,native_id,lat,lon,Elevation
60,12.0,NaN,BAKER CREEK,4973,52.963900,-122.831100,826.0
61,12.0,NaN,Begbie DL,509,51.473300,-121.360000,1276.0
62,137.0,NaN,Agassiz - Centential Park,T051,49.238400,-121.774800,23.0
63,5.0,NaN,Gold River Below Ucona River 2,GLD 2,49.703200,-126.109406,10.0
64,5.0,NaN,Rees Creek,REC,49.605361,-125.288167,280.0
65,14.0,NaN,July Mountain,1C45P,49.707640,-121.085470,1860.0
66,14.0,NaN,Paradise Lake,AC44P,49.948639,-120.282270,1640.0
67,14.0,NaN,Deadman River,1C32P,51.098510,-120.609690,1430.0
68,14.0,NaN,Penfold Creek,1C23P,52.748820,-120.560570,1740.0
69,14.0,NaN,Puntzi,1C22AP,52.109000,-124.152730,920.0


In [8]:
# 1. Rename df_pcds columns to match df_ted
df_pcds_renamed = df_pcds.rename(columns={
    'pcds_sation_name': 'Station_name',
    'pcds_native_id': 'native_id',
    'pcds_lat': 'lat',
    'pcds_lon': 'lon',
    'pcds_elev': 'Elevation'
})

df_pcds_renamed = df_pcds_renamed.drop(columns=['Station ID'])

merged = df_pcds_renamed.merge(
    df_ted,
    left_index=True,
    right_index=True,
    how='inner',
    suffixes=('_pcds', '_ted')
)

# 3. Compare key columns
merged['Station_name_match']  = merged['Station_name_pcds'].str.strip()  == merged['Station_name_ted'].str.strip()
merged['lat_match']  = merged['lat_pcds']  == merged['lat_ted']
merged['lon_match']  = merged['lon_pcds']  == merged['lon_ted']
merged['elev_match'] = merged['Elevation_pcds'] == merged['Elevation_ted']
merged['id_match']   = merged['native_id_pcds'].astype(str).str.strip() == merged['native_id_ted'].astype(str).str.strip()

# 4. Whether all match
merged['all_match'] = (
    merged['Station_name_match'] &
    merged['lat_match'] &
    merged['lon_match'] &
    merged['elev_match'] &
    merged['id_match']
)

# 5. Show results
merged[['Station_name_match', 'lat_match', 'lon_match', 'elev_match', 'id_match', 'all_match']]


# df_pcds_renamed.iloc[69]

# merged

,Station_name_match,lat_match,lon_match,elev_match,id_match,all_match
0,True,True,True,True,True,True
1,True,True,True,True,True,True
2,True,True,True,True,True,True
3,True,True,True,True,True,True
4,True,True,True,True,True,True
...,...,...,...,...,...,...
124,True,True,True,True,True,True
125,True,True,True,True,True,True
126,True,True,True,True,True,True
127,True,False,False,False,True,False


In [12]:
mismatched = merged[merged['all_match'] == False]
mismatched

mismatched.to_csv("Sheet1_Addtional_mismatch_44items.csv", index=False)
mismatched

,Network ID,Station_name_pcds,native_id_pcds,lat_pcds,lon_pcds,Elevation_pcds,Station_name_ted,native_id_ted,lat_ted,lon_ted,Elevation_ted,Station_name_match,lat_match,lon_match,elev_match,id_match,all_match
34,10.0,South ErickSon,AGCRESSEKN,49.080300,-116.493100,650.0,South Erickson,AGCRESSEKN,49.0803,-116.4931,650,False,True,True,True,True,False
48,2.0,Johnson Hill,14093,49.205280,-122.802020,93.0,Port Mann Bridge Johnson Hill,14093,49.20528,-122.80202,93,False,True,True,True,True,False
49,12.0,LITTLE CHOPAKA,3730,49.025100,-119.690900,461.0,LITTLE CHOPEKA,3810,49.0251,-119.6909,416,False,True,True,False,False,False
51,12.0,WILLIS,3873,49.339400,-120.407100,1455.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
52,12.0,BURNS LAKE,3991,54.259100,-125.759900,830.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
54,12.0,WILDHORSE WX,4250,51.233300,-122.150000,577.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
55,12.0,CHETWYND FB,4270,55.690700,-121.613700,616.0,CHETWYND,4270,55.6907,-121.6137,616,False,True,True,True,True,False
58,12.0,MESACHIE 3,4513,48.819900,-124.131000,197.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
59,12.0,MASHITER,4552,49.752000,-123.098300,262.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
60,12.0,BAKER CREEK,4973,52.963900,-122.831100,826.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False


In [10]:
len(mismatched)

44

### Going to using the info in the pcds sheet to insert these stations

In [160]:
df_pcds_renamed

save_path = './comparison_forms/'
df_pcds_renamed.to_csv(save_path + '2_129_additional_station_insert_info.csv', index=False)
df_pcds_renamed

,Network ID,Station_name,native_id,lat,lon,Elevation
0,10.0,Summerland Front Bench,AGOKNGSLFB,49.579400,-119.651000,426.0
1,10.0,Alkali Lake,AGAL001,51.790278,-121.251111,677.0
2,10.0,BCGPA Dawson Creek,AGBCPRDNCK,55.772100,-120.143000,673.0
3,10.0,Bearflats,AGBCPRBFLT,56.246100,-121.308000,504.0
4,10.0,Bella Vista,AGOKNGBVST,50.255100,-119.341300,437.0
...,...,...,...,...,...,...
124,137.0,Disappointment Lake,1D18P,49.555000,-122.759167,1050.0
125,137.0,Pallisade Lake,3A09P,49.454444,-123.031667,900.0
126,137.0,Orchid Lake,3A19P,49.536111,-123.050000,1180.0
127,137.0,Loch Lomond,3A29P,49.592330,-123.041450,1065.0


### Insert into db

In [163]:
from sqlalchemy import func

stations_created = []

# for _, row in df_pcds_renamed.iloc[0:2].iterrows():
for _, row in df_pcds_renamed.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Summerland Front Bench', 14843), ('Alkali Lake', 14844), ('BCGPA Dawson Creek', 14845), ('Bearflats', 14846), ('Bella Vista', 14847), ('Buick', 14848), ('Canyon', 14849), ('Cecil Lake', 14850), ('Central Erickson', 14851), ('Crescent Island', 14852), ('Farmington', 14853), ('Flatrock', 14854), ('Grandview Flats', 14855), ('Groundbirch', 14856), ('Hullcar Rd East', 14857), ('Hullcar Rd West', 14858), ('Kaleden', 14859), ('Kelly Lake', 14860), ('Keremeos Bipass', 14861), ('Lavington', 14862), ('Merville', 14863), ('Monte Creek', 14864), ('Montney - Bickfords', 14865), ('Naramata Bench', 14866), ('North Crawford Rd', 14867), ('North Erickson', 14868), ('North Grand Forks', 14869), ('North Lister (Riverview)', 14870), ('Osoyoos North', 14871), ('Osoyoos Upper Bench', 14872), ('Permberton Meadows', 14873), ('Prespatou', 14874), ('Rolla', 14875), ('Saanichton', 14876), ('South ErickSon', 14877), ('South Kelowna', 14878), ('Tomslake', 14879), ('Tower Lake', 14880), ('Tsawwassen', 

The insert time is 2025-12-19 10:43:29